# Capstone P3 — Experimento estatístico honesto e reproduzível

## Goal

Executar um protocolo completo para comparar, nas mesmas tarefas, um sistema de IA **baseline** e uma variante **governada**. A análise combina efeito primário, incerteza, relevância prática, *guardrails*, multiplicidade, sensibilidade e limites de conclusão.

Este notebook acompanha a [Aula 24](../aulas/24-capstone-experimento-estatistico-honesto.md). Os dados são **sintéticos**: demonstram o método, não o desempenho de um produto real.

## Context & Methods

### Protocolo congelado antes da geração

- **População-alvo didática:** tarefas de atendimento, jurídico, operações e pesquisa.
- **Unidade de análise:** tarefa; cada tarefa é executada nas duas variantes.
- **Estimando primário:** média de `qualidade_governado - qualidade_baseline`.
- **Hipótese primária:** diferença média igual a zero versus diferente de zero.
- **SESOI:** 0,02 ponto de qualidade.
- **Amostra:** 240 tarefas, 60 por domínio.
- **Ordem:** variante que executa primeiro é randomizada por tarefa.
- **IC primário:** t pareado de 95%, reconciliado com bootstrap por tarefa.
- **Teste primário:** permutação pareada por troca de sinais, bilateral.
- **Família secundária:** latência, custo e erro crítico; ajuste de Holm.
- **Guardrails:** aumento médio de latência < 450 ms e de custo < 0,008/tarefa.
- **Regra:** IC primário acima de zero, estimativa ≥ SESOI, guardrails aprovados e sem aumento pontual de erro crítico.

Nenhuma regra abaixo é alterada depois da geração dos dados.

### Setup

**Dependências:** Python 3.11+, NumPy 2+, pandas 2+, Matplotlib 3.8+ e SciPy 1.13+.

```python
%pip install "numpy>=2.0" "pandas>=2.0" "matplotlib>=3.8" "scipy>=1.13"
```

O laboratório não usa rede, credenciais nem arquivos externos. A seed e as versões são registradas.

In [ ]:
import hashlib
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats

SEED = 20260908
rng = np.random.default_rng(SEED)

SESOI_QUALIDADE = 0.02
LIMITE_LATENCIA_MS = 450.0
LIMITE_CUSTO = 0.008
ALPHA = 0.05

print(f"Python {platform.python_version()}")
print(f"NumPy {np.__version__} | pandas {pd.__version__}")
print(f"Matplotlib {matplotlib.__version__} | SciPy {scipy.__version__}")
print(f"Seed: {SEED}")

## Data

### 1. Gere a amostra sintética documentada

O efeito de tarefa é compartilhado pelas variantes, criando o pareamento. Dificuldade, domínio e ordem são registrados. Em um experimento real, esta célula seria substituída por uma leitura imutável e versionada.

In [ ]:
n_tarefas = 240
dominios = np.repeat(["atendimento", "juridico", "operacoes", "pesquisa"], n_tarefas // 4)
rng.shuffle(dominios)
dificuldade = rng.beta(2.2, 2.0, size=n_tarefas)
governado_primeiro = rng.random(n_tarefas) < 0.5

efeito_dominio_base = {
    "atendimento": 0.025,
    "juridico": -0.025,
    "operacoes": 0.000,
    "pesquisa": 0.015,
}
efeito_dominio_delta = {
    "atendimento": 0.005,
    "juridico": 0.000,
    "operacoes": -0.004,
    "pesquisa": 0.008,
}

base_dominio = np.array([efeito_dominio_base[d] for d in dominios])
delta_dominio = np.array([efeito_dominio_delta[d] for d in dominios])
qualidade_baseline = np.clip(
    0.72 + base_dominio - 0.20 * (dificuldade - 0.5) + rng.normal(0, 0.065, n_tarefas),
    0.20,
    0.98,
)
delta_latente = (
    0.034 - 0.012 * dificuldade + delta_dominio
    + np.where(governado_primeiro, 0.002, -0.002)
    + rng.normal(0, 0.035, n_tarefas)
)
qualidade_governado = np.clip(qualidade_baseline + delta_latente, 0.0, 1.0)

latencia_baseline_ms = rng.lognormal(mean=np.log(900), sigma=0.22, size=n_tarefas)
latencia_governado_ms = np.maximum(
    1,
    latencia_baseline_ms + 275 + 110 * dificuldade + rng.normal(0, 85, n_tarefas),
)
custo_baseline = np.maximum(0, 0.012 + 0.004 * dificuldade + rng.normal(0, 0.0012, n_tarefas))
custo_governado = np.maximum(
    0,
    custo_baseline + 0.0052 + 0.001 * dificuldade + rng.normal(0, 0.0010, n_tarefas),
)

prob_erro_base = np.clip(0.16 + 0.14 * dificuldade, 0, 1)
prob_erro_gov = np.clip(prob_erro_base - 0.055, 0, 1)
erro_baseline = rng.random(n_tarefas) < prob_erro_base
erro_governado = rng.random(n_tarefas) < prob_erro_gov

dados = pd.DataFrame({
    "task_id": [f"T{i:03d}" for i in range(1, n_tarefas + 1)],
    "dominio": dominios,
    "dificuldade": dificuldade,
    "governado_primeiro": governado_primeiro,
    "qualidade_baseline": qualidade_baseline,
    "qualidade_governado": qualidade_governado,
    "latencia_baseline_ms": latencia_baseline_ms,
    "latencia_governado_ms": latencia_governado_ms,
    "custo_baseline": custo_baseline,
    "custo_governado": custo_governado,
    "erro_critico_baseline": erro_baseline.astype(int),
    "erro_critico_governado": erro_governado.astype(int),
})

dados["delta_qualidade"] = dados["qualidade_governado"] - dados["qualidade_baseline"]
dados["delta_latencia_ms"] = dados["latencia_governado_ms"] - dados["latencia_baseline_ms"]
dados["delta_custo"] = dados["custo_governado"] - dados["custo_baseline"]
dados["delta_erro_critico"] = dados["erro_critico_governado"] - dados["erro_critico_baseline"]

print(dados.head(6).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

### 2. Registre proveniência e impressão digital

O hash abaixo identifica exatamente a tabela analítica produzida por esta execução. Um projeto real deve também preservar manifesto, origem, licença, configuração das variantes e hashes dos dados brutos.

In [ ]:
csv_canonico = dados.to_csv(index=False, float_format="%.10g", lineterminator="\n")
sha256_dados = hashlib.sha256(csv_canonico.encode("utf-8")).hexdigest()

manifesto = {
    "fonte": "geração sintética documentada no notebook",
    "unidade": "tarefa",
    "n_tarefas": len(dados),
    "seed": SEED,
    "sha256_tabela_analitica": sha256_dados,
    "gerador": "numpy.random.Generator/PCG64",
}
print(pd.Series(manifesto).to_string())

### 3. Valide antes de estimar o efeito

Estas checagens refletem o contrato do protocolo. Elas não removem casos com base no desfecho.

In [ ]:
colunas_obrigatorias = {
    "task_id", "dominio", "dificuldade", "governado_primeiro",
    "qualidade_baseline", "qualidade_governado",
    "latencia_baseline_ms", "latencia_governado_ms",
    "custo_baseline", "custo_governado",
    "erro_critico_baseline", "erro_critico_governado",
}

validacoes = {
    "schema": colunas_obrigatorias.issubset(dados.columns),
    "240 tarefas": len(dados) == 240,
    "task_id único": dados["task_id"].is_unique,
    "sem ausências": not dados[list(colunas_obrigatorias)].isna().any().any(),
    "qualidade em [0,1]": dados.filter(regex="^qualidade_").stack().between(0, 1).all(),
    "dificuldade em [0,1]": dados["dificuldade"].between(0, 1).all(),
    "latência positiva": (dados.filter(regex="^latencia_") > 0).all().all(),
    "custo não negativo": (dados.filter(regex="^custo_") >= 0).all().all(),
    "60 por domínio": dados["dominio"].value_counts().eq(60).all(),
    "ordem aproximadamente balanceada": 0.35 <= dados["governado_primeiro"].mean() <= 0.65,
}

print(pd.Series(validacoes, name="aprovada").to_string())
assert all(validacoes.values())

## Results

### 4. Descreva diferenças pareadas

O foco é a distribuição das diferenças por tarefa. Médias isoladas de cada variante escondem a correlação criada pelo pareamento.

In [ ]:
resumo_pareado = dados[[
    "delta_qualidade", "delta_latencia_ms", "delta_custo", "delta_erro_critico"
]].agg(["count", "mean", "std", "median", "min", "max"]).T

print(resumo_pareado.to_string(float_format=lambda x: f"{x:.6f}"))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(dados["delta_qualidade"], bins=24, color="#2563eb", alpha=0.82)
axes[0].axvline(0, color="black", linestyle="--", label="sem diferença")
axes[0].axvline(SESOI_QUALIDADE, color="#dc2626", linestyle=":", label="SESOI=0,02")
axes[0].set(xlabel="Governado − baseline", ylabel="Tarefas", title="Diferença pareada de qualidade")
axes[0].legend()

axes[1].scatter(dados["qualidade_baseline"], dados["qualidade_governado"], s=18, alpha=0.55)
axes[1].plot([0.5, 1], [0.5, 1], color="black", linestyle="--")
axes[1].set(xlabel="Qualidade baseline", ylabel="Qualidade governado", title="Cada ponto é a mesma tarefa")
axes[1].set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()

### 5. Estime efeito, IC t e tamanho padronizado

O efeito bruto é a quantidade principal para decisão. `d_z` usa o desvio-padrão das diferenças, não o desvio de cada variante separadamente.

In [ ]:
diferencas = dados["delta_qualidade"].to_numpy()
n = len(diferencas)
media_delta = diferencas.mean()
dp_delta = diferencas.std(ddof=1)
se_delta = dp_delta / np.sqrt(n)
t_critico = stats.t.ppf(1 - ALPHA / 2, df=n - 1)
ic_t = (media_delta - t_critico * se_delta, media_delta + t_critico * se_delta)
d_z = media_delta / dp_delta
teste_t = stats.ttest_rel(dados["qualidade_governado"], dados["qualidade_baseline"])

print(f"Diferença média={media_delta:.9f}")
print(f"IC t 95%=[{ic_t[0]:.9f}; {ic_t[1]:.9f}]")
print(f"Erro-padrão={se_delta:.9f}; d_z={d_z:.6f}")
print(f"t={teste_t.statistic:.6f}; p={teste_t.pvalue:.3e}")

### 6. Reconcile com bootstrap por tarefa

Cada reamostra sorteia tarefas completas, preservando as duas variantes. O intervalo percentil não substitui a discussão do desenho; oferece uma checagem de sensibilidade à aproximação t.

In [ ]:
B_BOOT = 10_000
indices_boot = rng.integers(0, n, size=(B_BOOT, n))
medias_boot = diferencas[indices_boot].mean(axis=1)
ic_boot = tuple(np.quantile(medias_boot, [ALPHA / 2, 1 - ALPHA / 2]))

print(f"Bootstrap percentil 95%=[{ic_boot[0]:.9f}; {ic_boot[1]:.9f}]")
print(f"Desvio das médias bootstrap={medias_boot.std(ddof=1):.9f}")

### 7. Execute permutação pareada

Sob o mecanismo nulo, os sinais das diferenças são trocáveis. Usamos correção `+1` no numerador e denominador para a aproximação Monte Carlo.

In [ ]:
B_PERM = 20_000
sinais = rng.choice(np.array([-1.0, 1.0]), size=(B_PERM, n))
estatisticas_nulas = (sinais * diferencas).mean(axis=1)
p_permutacao = (1 + np.count_nonzero(np.abs(estatisticas_nulas) >= abs(media_delta))) / (B_PERM + 1)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.hist(estatisticas_nulas, bins=45, color="#94a3b8", alpha=0.85, label="hipótese nula")
ax.axvline(media_delta, color="#dc2626", linewidth=2, label="efeito observado")
ax.axvline(-media_delta, color="#dc2626", linewidth=1.5, linestyle="--")
ax.set(xlabel="Diferença média permutada", ylabel="Frequência", title="Teste de permutação pareado")
ax.legend()
plt.tight_layout()
plt.show()

print(f"p-value bilateral de permutação={p_permutacao:.8f}")

### 8. Analise métricas secundárias com Holm

Latência e custo usam testes pareados. Para erro crítico binário, o teste exato considera apenas tarefas discordantes entre variantes.

In [ ]:
def ajuste_holm(p_values):
    p = np.asarray(p_values, dtype=float)
    if p.ndim != 1 or np.any((p < 0) | (p > 1)):
        raise ValueError("p-values devem formar vetor em [0,1]")
    ordem = np.argsort(p)
    ordenados = p[ordem]
    m = len(p)
    ajustados_ordenados = np.maximum.accumulate((m - np.arange(m)) * ordenados)
    ajustados_ordenados = np.minimum(1.0, ajustados_ordenados)
    ajustados = np.empty_like(ajustados_ordenados)
    ajustados[ordem] = ajustados_ordenados
    return ajustados


teste_latencia = stats.ttest_rel(dados["latencia_governado_ms"], dados["latencia_baseline_ms"])
teste_custo = stats.ttest_rel(dados["custo_governado"], dados["custo_baseline"])

base_erra_gov_acerta = int(((dados["erro_critico_baseline"] == 1) & (dados["erro_critico_governado"] == 0)).sum())
base_acerta_gov_erra = int(((dados["erro_critico_baseline"] == 0) & (dados["erro_critico_governado"] == 1)).sum())
discordantes = base_erra_gov_acerta + base_acerta_gov_erra
teste_mcnemar = stats.binomtest(base_acerta_gov_erra, n=discordantes, p=0.5, alternative="two-sided")

secundarias = pd.DataFrame({
    "métrica": ["latência_ms", "custo", "erro_crítico"],
    "efeito_governado_menos_base": [
        dados["delta_latencia_ms"].mean(),
        dados["delta_custo"].mean(),
        dados["delta_erro_critico"].mean(),
    ],
    "p_raw": [teste_latencia.pvalue, teste_custo.pvalue, teste_mcnemar.pvalue],
})
secundarias["p_holm"] = ajuste_holm(secundarias["p_raw"])
secundarias["rejeita_0_05"] = secundarias["p_holm"] < ALPHA

print(f"Discordantes: baseline erra/governado acerta={base_erra_gov_acerta}; ")
print(f"              baseline acerta/governado erra={base_acerta_gov_erra}")
print(secundarias.to_string(index=False, float_format=lambda x: f"{x:.8f}"))

### 9. Faça análises de sensibilidade

Média aparada e mediana reduzem influência de extremos. O efeito por ordem procura um padrão incompatível com a randomização. Essas verificações foram pré-especificadas, mas não substituem a métrica primária.

In [ ]:
media_aparada = stats.trim_mean(diferencas, proportiontocut=0.10)
mediana_delta = np.median(diferencas)

por_ordem = (
    dados.groupby("governado_primeiro", observed=True)["delta_qualidade"]
    .agg(["count", "mean", "std"])
    .rename(index={False: "baseline primeiro", True: "governado primeiro"})
)
teste_ordem = stats.ttest_ind(
    dados.loc[dados["governado_primeiro"], "delta_qualidade"],
    dados.loc[~dados["governado_primeiro"], "delta_qualidade"],
    equal_var=False,
)

print(f"Média convencional={media_delta:.9f}")
print(f"Média aparada 10%={media_aparada:.9f}")
print(f"Mediana={mediana_delta:.9f}")
print()
print(por_ordem.to_string(float_format=lambda x: f"{x:.6f}"))
print(f"Diferença por ordem, Welch p={teste_ordem.pvalue:.6f}")

### 10. Explore heterogeneidade por domínio

Os intervalos por domínio são exploratórios: o estudo foi dimensionado para o efeito médio, não para confirmar quatro efeitos separados.

In [ ]:
linhas_dominio = []
for dominio, grupo in dados.groupby("dominio", sort=True):
    valores = grupo["delta_qualidade"].to_numpy()
    media = valores.mean()
    se = valores.std(ddof=1) / np.sqrt(len(valores))
    tcrit = stats.t.ppf(0.975, len(valores) - 1)
    linhas_dominio.append({
        "dominio": dominio,
        "n": len(valores),
        "media": media,
        "ic_inf": media - tcrit * se,
        "ic_sup": media + tcrit * se,
    })

por_dominio = pd.DataFrame(linhas_dominio)
print(por_dominio.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

fig, ax = plt.subplots(figsize=(7.5, 4.4))
y = np.arange(len(por_dominio))
ax.errorbar(
    por_dominio["media"], y,
    xerr=[por_dominio["media"] - por_dominio["ic_inf"], por_dominio["ic_sup"] - por_dominio["media"]],
    fmt="o", color="#2563eb", capsize=4,
)
ax.axvline(0, color="black", linestyle="--")
ax.axvline(SESOI_QUALIDADE, color="#dc2626", linestyle=":", label="SESOI")
ax.set(yticks=y, yticklabels=por_dominio["dominio"], xlabel="Diferença média de qualidade (IC 95%)", title="Heterogeneidade exploratória")
ax.legend()
plt.tight_layout()
plt.show()

### 11. Aplique a regra de decisão registrada

Uma conclusão estatística positiva não basta. A tabela separa os critérios e torna explícito por que a decisão foi tomada.

In [ ]:
delta_latencia = dados["delta_latencia_ms"].mean()
delta_custo = dados["delta_custo"].mean()
delta_erro = dados["delta_erro_critico"].mean()

criterios = pd.Series({
    "IC primário acima de zero": ic_t[0] > 0,
    "estimativa atinge SESOI": media_delta >= SESOI_QUALIDADE,
    "latência dentro do limite": delta_latencia < LIMITE_LATENCIA_MS,
    "custo dentro do limite": delta_custo < LIMITE_CUSTO,
    "sem aumento pontual de erro crítico": delta_erro <= 0,
})
decisao = "prosseguir para piloto controlado" if criterios.all() else "não adotar sem nova evidência/ajuste"

print(criterios.rename("aprovado").to_string())
print()
print("Decisão:", decisao)
print(f"Qualidade: {media_delta:.6f}, IC [{ic_t[0]:.6f}; {ic_t[1]:.6f}], SESOI={SESOI_QUALIDADE:.3f}")
print(f"Latência: +{delta_latencia:.3f} ms (limite < {LIMITE_LATENCIA_MS:.0f})")
print(f"Custo: +{delta_custo:.6f} (limite < {LIMITE_CUSTO:.3f})")
print(f"Erro crítico: {delta_erro:+.6f} por tarefa")

## Checks

As verificações finais reconciliam protocolo, cálculos e interpretação. Limites largos para resultados sintéticos testam propriedades; números centrais são impressos para auditoria.

In [ ]:
assert len(dados) == 240 and dados["task_id"].is_unique
assert dados["dominio"].value_counts().eq(60).all()
assert np.isclose(media_delta, dados["qualidade_governado"].mean() - dados["qualidade_baseline"].mean())
assert np.isclose(teste_t.statistic, media_delta / se_delta)
assert ic_t[0] < media_delta < ic_t[1]
assert ic_boot[0] < media_delta < ic_boot[1]
assert abs((medias_boot.std(ddof=1) / se_delta) - 1) < 0.08
assert 0 < p_permutacao <= 1
assert np.all(secundarias["p_holm"] >= secundarias["p_raw"])
assert media_aparada > 0 and mediana_delta > 0
assert por_dominio["n"].eq(60).all()
assert len(sha256_dados) == 64
assert criterios.all()

print("Todas as verificações foram aprovadas.")
print(f"Dados: n={n}; SHA-256={sha256_dados}")
print(f"Qualidade: Δ={media_delta:.6f}; IC t=[{ic_t[0]:.6f}; {ic_t[1]:.6f}]; d_z={d_z:.6f}")
print(f"Bootstrap=[{ic_boot[0]:.6f}; {ic_boot[1]:.6f}]; permutação p={p_permutacao:.8f}")
print(f"Guardrails: latência={delta_latencia:.3f} ms; custo={delta_custo:.6f}; erro={delta_erro:+.6f}")
print(f"Sensibilidade: aparada={media_aparada:.6f}; mediana={mediana_delta:.6f}; ordem p={teste_ordem.pvalue:.6f}")
print("Decisão:", decisao)

## Takeaways

- O desenho pareado usa a tarefa como unidade e estima diretamente `governado − baseline`.
- O efeito primário deve ser lido junto ao IC, à SESOI e aos *guardrails*.
- Bootstrap e permutação reamostram/trocam no nível da tarefa, preservando o desenho.
- Holm controla a família de métricas secundárias planejadas.
- Resultados por domínio são exploratórios e precisam de confirmação independente.
- O hash registra a tabela analítica produzida, mas não substitui um manifesto real de origem e transformação.
- **Este resultado não prova que a variante governada será superior em outros domínios, populações, modelos ou condições operacionais.**

## Next Steps

1. Substitua a geração sintética por um dataset versionado e preserve o protocolo.
2. Defina rubrica, avaliadores, cegamento e política de falhas antes da coleta.
3. Se houver usuários ou sessões repetidas, adapte reamostragem e modelo à estrutura de grupos.
4. Publique relatório, hashes, dependências e limitações junto ao commit analisado.
5. Continue para [Machine Learning — Aula 03: pré-processamento, pipelines e data leakage](../../03-machine-learning/aulas/03-preprocessamento-pipelines-leakage.md).